# 0. Setup and data

This notebook installs the dependencies, downloads each benchmark, and checks that
every dataset loads through the same code path the experiments use. Run it once.
Everything lands under `data/raw/`. Start from the repository root.


## Dependencies


In [ ]:
!pip install -q -r ../requirements.txt


## Download and assemble the datasets

`prepare_data.py` fetches the public raw files and reconstructs the processed
benchmarks. The output is numerically identical to the files used in the paper
(IHDP is byte-exact; Twins and LaLonde match on every numeric column).
IHDP, Twins, and LaLonde are fully automated:


In [ ]:
import subprocess
subprocess.run(['python', 'data/prepare_data.py', 'all'], cwd='..', check=True)


### ACIC 2016

ACIC needs the official R package. See `DATA.md` for the exact steps; briefly, in R:

```r
R CMD INSTALL aciccomp-master/2016   # from a clone of github.com/vdorie/aciccomp
library(aciccomp2016)
sim <- dgp_2016(input_2016, parameters = 7, random.seed = 1)
```

Export `x.csv`, `z_dgp7.csv`, `y_dgp7.csv`, and `potential_outcomes_dgp7.csv` into
`data/raw/acic/` (one-hot the three factor columns to 82 numeric covariates).


### ACS

folktables downloads the California 2018 person file on first use:


In [ ]:
from folktables import ACSDataSource
# writes data/2018/1-Year/psam_p06.csv under the repo root
ACSDataSource(survey_year='2018', horizon='1-Year', survey='person').get_data(states=['CA'], download=True)


## Verify

This loads every dataset through the experiment driver and asserts none falls back
to a synthetic surrogate. You should see a real source, `n`, `d`, and `tau_true` for
each one.


In [ ]:
import sys; sys.path.insert(0, '..')
import subprocess
subprocess.run(['python', 'data/verify_provenance.py'], cwd='..', check=True)
